# ML-008 — Movie Review Sentiment Classification
**Domain:** Entertainment / NLP  |  **Type:** Binary classification (Positive / Negative)
**Platform:** Google Colab  |  **Dataset:** 10,000 IMDB reviews (5,000 positive + 5,000 negative)

**Pipeline:** Load data → Clean text → TF-IDF → Train models → Evaluate → Custom tests → Error analysis → Save → Live demo

| Part | Owner | Steps |
|---|---|---|
| Data mining and cleaning | Member 1 | Steps 2–4 |
| Features and models | Member 2 | Steps 5–6 and 12 (Gradio demo) |
| Evaluation and analysis | Member 3 | Steps 7–10 |
| Save and share | Member 2 + 3 | Step 11 |

> Run the cells top to bottom (Shift+Enter). Final run: **Runtime → Restart session and run all**.

## Step 0 — Connect Google Drive (Colab only)
If you are not in Colab (local Jupyter), this cell just skips.

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
print("Running in Colab:", IN_COLAB)

## Step 1 — Imports and settings

In [ ]:
import os
import re
import shutil
import warnings
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, confusion_matrix, classification_report)

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
RANDOM_STATE = 42
os.makedirs("outputs", exist_ok=True)      # all files for the team are saved here
print("Setup done")

## Step 2 — Load the dataset  *(Member 1)*
Set the exact path of your CSV between the quotes. Tip: in the left panel click the file → three dots → **Copy path**.
The file must have the columns `review` and `sentiment` (positive / negative).

`SAMPLE_PER_CLASS = 5000` means 5,000 positive + 5,000 negative = **10,000 rows**. **All team members must use the same value.**

In [ ]:
DATA_PATH = "/content/drive/MyDrive/Colab Notebooks/10,000 DATASET (1).csv"
SAMPLE_PER_CLASS = 5000

if not os.path.exists(DATA_PATH):
    raise FileNotFoundError(
        f"Dataset not found at: {DATA_PATH}\n"
        "Check the path (left panel > file > Copy path) and that it is inside quotes.")

df = pd.read_csv(DATA_PATH)
print("Loaded CSV:", df.shape)

missing = {"review", "sentiment"} - set(df.columns)
if missing:
    raise KeyError(f"Missing column(s): {missing}. Found: {df.columns.tolist()}")

df["sentiment"] = df["sentiment"].astype(str).str.strip().str.lower()
df = df[["review", "sentiment"]].dropna().drop_duplicates().reset_index(drop=True)
print(df["sentiment"].value_counts().to_dict())

# Balanced sample
n = min(SAMPLE_PER_CLASS, df["sentiment"].value_counts().min())
df = pd.concat([df[df["sentiment"] == "positive"].sample(n, random_state=RANDOM_STATE),
                df[df["sentiment"] == "negative"].sample(n, random_state=RANDOM_STATE)])
df = df.sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)
df["label"] = (df["sentiment"] == "positive").astype(int)   # 1 = positive, 0 = negative
print("Working dataset:", df.shape)
df.head()

## Step 3 — Quick exploration (EDA)  *(Member 1)*

In [ ]:
print(df["sentiment"].value_counts())
df["word_count"] = df["review"].str.split().str.len()
print(df["word_count"].describe().round(1))

fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))
sns.countplot(data=df, x="sentiment", palette=["#d9534f", "#2e8b57"], ax=ax[0])
ax[0].set_title("Class balance")
sns.histplot(data=df, x="word_count", hue="sentiment", bins=40, ax=ax[1])
ax[1].set_title("Review length (words)")
ax[1].set_xlim(0, 1000)
plt.tight_layout()
plt.savefig("outputs/eda_charts.png", dpi=150, bbox_inches="tight")
plt.show()

## Step 4 — Text preprocessing  *(Member 1)*
Steps: lowercase → remove HTML tags → remove special characters and digits → remove extra spaces → tokenize → remove stopwords.

We **keep** negation words (`not`, `no`, `nor`, `never`) because "not good" and "good" have opposite meanings.

In [ ]:
KEEP = {"not", "no", "nor", "never", "cannot", "n't"}
STOPWORDS = set(ENGLISH_STOP_WORDS) - KEEP

def clean_text(text):
    text = text.lower()                                # 1. lowercase
    text = re.sub(r"<[^>]+>", " ", text)               # 2. remove HTML tags
    text = re.sub(r"[^a-z\s]", " ", text)              # 3. remove special characters / digits
    text = re.sub(r"\s+", " ", text).strip()           # 4. remove extra spaces
    tokens = text.split()                              # 5. tokenization
    tokens = [t for t in tokens if t not in STOPWORDS and len(t) > 1]   # 6. stopword removal
    return " ".join(tokens)

sample = "<br />The movie was NOT good!!! 10/10 acting?? but the story was boring."
print("RAW  :", sample)
print("CLEAN:", clean_text(sample))

In [ ]:
df["clean_review"] = df["review"].apply(clean_text)

print("Empty cleaned reviews:", (df["clean_review"].str.len() == 0).sum())
print("Null values:", df.isnull().sum().sum())

# Before / after examples for the report
for i in range(3):
    print("BEFORE:", df["review"][i][:150].replace("\n", " "))
    print("AFTER :", df["clean_review"][i][:150])
    print("-" * 80)

## Step 5 — Train/test split and TF-IDF  *(Member 2)*
The test set is locked away here. TF-IDF is fitted on the **training** data only (prevents data leakage).

In [ ]:
X_train_text, X_test_text, y_train, y_test = train_test_split(
    df["clean_review"], df["label"],
    test_size=0.2, random_state=RANDOM_STATE, stratify=df["label"])
print("Train:", len(X_train_text), y_train.value_counts().to_dict())
print("Test :", len(X_test_text), y_test.value_counts().to_dict())

tfidf = TfidfVectorizer(max_features=5000, ngram_range=(1, 2), min_df=2)
X_train = tfidf.fit_transform(X_train_text)
X_test = tfidf.transform(X_test_text)

print("Train matrix:", X_train.shape, "| Test matrix:", X_test.shape)
print("Sample features:", tfidf.get_feature_names_out()[:15])

## Step 6 — Train the models  *(Member 2)*

In [ ]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
    "Naive Bayes": MultinomialNB(),
    "Linear SVM": LinearSVC(random_state=RANDOM_STATE),
}

results, preds = [], {}
for name, model in models.items():
    model.fit(X_train, y_train)          # TRAIN
    p = model.predict(X_test)            # PREDICT the unseen test reviews
    preds[name] = p
    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, p),
        "Precision": precision_score(y_test, p),
        "Recall": recall_score(y_test, p),
        "F1-Score": f1_score(y_test, p),
    })

results_df = pd.DataFrame(results).set_index("Model").round(4)
results_df

In [ ]:
# Overfitting check: a big gap between train and test means the model memorised the training data
for name, model in models.items():
    print(f"{name:20s} train={model.score(X_train, y_train):.3f}   test={model.score(X_test, y_test):.3f}")

results_df.plot(kind="bar", figsize=(9, 4), colormap="viridis")
plt.title("Model comparison"); plt.ylim(0.6, 1.0); plt.xticks(rotation=0)
plt.legend(loc="lower right"); plt.tight_layout()
plt.savefig("outputs/model_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

best_name = results_df["F1-Score"].idxmax()
best_model = models[best_name]
print("Best model:", best_name)

**Optional (Member 2):** a more reliable comparison with 5-fold cross-validation, and a small tuning table. Skip if short on time.

In [ ]:
# Optional A: 5-fold cross-validation on the training data
for name, model in models.items():
    s = cross_val_score(model, X_train, y_train, cv=5, scoring="f1")
    print(f"{name:20s} mean F1 = {s.mean():.4f}  (+/- {s.std():.4f})")

In [ ]:
# Optional B: tuning Logistic Regression C (record these in the report; gains are small)
for C in [0.5, 1, 5, 10]:
    m = LogisticRegression(C=C, max_iter=1000, random_state=RANDOM_STATE).fit(X_train, y_train)
    print("C =", C, " F1 =", round(f1_score(y_test, m.predict(X_test)), 4))

## Step 7 — Detailed evaluation of the best model  *(Member 3)*

In [ ]:
y_pred = preds[best_name]
print(classification_report(y_test, y_pred, target_names=["Negative", "Positive"]))

cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["Negative", "Positive"], yticklabels=["Negative", "Positive"])
plt.xlabel("Predicted"); plt.ylabel("Actual"); plt.title(f"Confusion Matrix - {best_name}")
plt.tight_layout()
plt.savefig("outputs/confusion_matrix.png", dpi=150, bbox_inches="tight")
plt.show()

tn, fp, fn, tp = cm.ravel()
print(f"TN={tn}  FP={fp}  FN={fn}  TP={tp}")

## Step 8 — Which words drive the prediction?  *(Member 3)*

In [ ]:
lr = models["Logistic Regression"]
feat = np.array(tfidf.get_feature_names_out())
coef = lr.coef_[0]
order = np.argsort(coef)
top_neg, top_pos = feat[order[:15]], feat[order[-15:]][::-1]
print("Top NEGATIVE words:", list(top_neg))
print("Top POSITIVE words:", list(top_pos))

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].barh(top_pos[::-1], np.sort(coef)[-15:], color="#2e8b57"); ax[0].set_title("Top positive words")
ax[1].barh(top_neg[::-1], np.sort(coef)[:15][::-1], color="#d9534f"); ax[1].set_title("Top negative words")
plt.tight_layout()
plt.savefig("outputs/top_words.png", dpi=150, bbox_inches="tight")
plt.show()

## Step 9 — Real-time testing with custom reviews  *(Member 3)*
The first 6 are from the project brief. **Add at least 5 of your own** (include a sarcastic one and a negation).

In [ ]:
def predict_sentiment(review, model=best_model):
    vec = tfidf.transform([clean_text(review)])
    return "Positive" if model.predict(vec)[0] == 1 else "Negative"

# (review, expected)  -> ADD YOUR OWN LINES HERE
custom_reviews = [
    ("The movie had excellent acting and a fantastic story.", "Positive"),
    ("The story was weak and the movie was extremely boring.", "Negative"),
    ("A brilliant film with a touching ending, I loved every minute.", "Positive"),
    ("Terrible script, wooden acting and a complete waste of time.", "Negative"),
    ("It was not good at all.", "Negative"),
    ("Not bad at all, I actually enjoyed it.", "Positive"),
    ("Oh great, another two hours of my life wasted.", "Negative"),
]

rows = [(r, exp, predict_sentiment(r)) for r, exp in custom_reviews]
custom_df = pd.DataFrame(rows, columns=["review", "expected", "predicted"])
custom_df["correct"] = custom_df["expected"] == custom_df["predicted"]
custom_df.to_csv("outputs/custom_tests.csv", index=False)
print(f"Correct: {custom_df['correct'].sum()} of {len(custom_df)}")
custom_df

In [ ]:
# Mixed / ambiguous reviews: no single right answer, just see what the model says and discuss it
for r in ["Great acting but a terrible story.", "I expected more, but it was okay I guess."]:
    print(f"{predict_sentiment(r):9s} | {r}")

## Step 10 — Analysis of incorrect predictions  *(Member 3)*
Look at the reviews the model got wrong and note **why** (sarcasm, negation, mixed opinions, quoted criticism, neutral description).

In [ ]:
err = X_test_text.to_frame("clean_review").copy()
err["actual"] = np.where(y_test == 1, "Positive", "Negative")
err["predicted"] = np.where(y_pred == 1, "Positive", "Negative")
err["original"] = df.loc[err.index, "review"]
err["wrong"] = err["actual"] != err["predicted"]
wrong = err[err["wrong"]]
print(f"Misclassified: {len(wrong)} of {len(err)}  ({len(wrong)/len(err):.1%})")
print(wrong.groupby(["actual", "predicted"]).size().rename("count"))

# Do long reviews fail more often?
long_mask = df.loc[err.index, "word_count"] > df["word_count"].quantile(0.75)
print("Error rate - longest 25% of reviews:", round(err["wrong"][long_mask].mean(), 3),
      "| other reviews:", round(err["wrong"][~long_mask].mean(), 3))

for _, row in wrong.sample(min(8, len(wrong)), random_state=RANDOM_STATE).iterrows():
    print(f"ACTUAL: {row['actual']:8s} | PREDICTED: {row['predicted']:8s}")
    print(row["original"][:300].replace("\n", " "), "...")
    print("-" * 90)

In [ ]:
# Export the errors; fill the 'reason' column by hand (sarcasm / negation / mixed opinion / quoted criticism / neutral)
export = wrong[["original", "actual", "predicted"]].copy()
export["reason"] = ""
export.to_csv("outputs/misclassified_reviews.csv", index=False)
print("Saved outputs/misclassified_reviews.csv")

### Write your observations here (for the report)
- Common error pattern 1: ...
- Common error pattern 2: ...
- Improvement ideas: bigger sample, lemmatization, tuning `C`, trying n-grams up to 3, or a transformer model such as BERT.

## Step 11 — Save the models and results, and download  *(Member 2 + 3)*
Creates `outputs/` and a zip. In Colab the zip downloads to your computer.

In [ ]:
joblib.dump(tfidf, "outputs/tfidf.pkl")
joblib.dump(best_model, "outputs/best_model.pkl")
joblib.dump(models["Logistic Regression"], "outputs/logistic_regression.pkl")
results_df.to_csv("outputs/model_results.csv")
df[["review", "clean_review", "sentiment"]].to_csv("outputs/cleaned_movie_reviews.csv", index=False)

shutil.make_archive("ML008_outputs", "zip", "outputs")
print("Files in outputs/:", sorted(os.listdir("outputs")))

try:
    from google.colab import files
    files.download("ML008_outputs.zip")
except ImportError:
    print("Not in Colab: the zip ML008_outputs.zip is in the notebook folder.")

## Step 12 — Live demo with Gradio  *(Member 2)*
Opens a small web page with a text box. In Colab it also prints a public link (works while the session runs, expires after a few days). Start it shortly before presenting.

It uses **Logistic Regression** because it can give probabilities (Linear SVM cannot).

In [ ]:
RUN_DEMO = True      # set to False to skip the demo

if RUN_DEMO:
    %pip install -q gradio
    import gradio as gr

    demo_model = models["Logistic Regression"]

    def predict(review):
        if not review or not review.strip():
            return {}
        p = demo_model.predict_proba(tfidf.transform([clean_text(review)]))[0]
        return {"Negative": float(p[0]), "Positive": float(p[1])}

    demo = gr.Interface(
        fn=predict,
        inputs=gr.Textbox(lines=4, label="Type a movie review"),
        outputs=gr.Label(num_top_classes=2, label="Sentiment"),
        title="Movie Review Sentiment Classifier",
        description="ML-008: TF-IDF + Logistic Regression trained on IMDB movie reviews.",
        examples=[["The movie had excellent acting and a fantastic story."],
                  ["The story was weak and the movie was extremely boring."]],
    )
    demo.launch(share=True)

## Conclusion
Fill in after the final run: best model, accuracy, F1-score, confusion-matrix insight, and the main reason for misclassifications.